# Alicorp - Modelo de Potencial Incremental de Ventas

**Caso:** predecir la probabilidad de que un cliente (bodega / puesto de mercado) tenga potencial de incremental de ventas.

**Estructura del notebook:**
1. Configuracion y carga de datos
2. Calidad de datos y preprocesamiento
3. Analisis exploratorio (EDA)
4. Feature engineering transaccional
5. Modelado y validacion cruzada
6. Evaluacion del modelo final
7. Analisis de impacto economico
8. Estrategia comercial por tier
9. Conclusiones y siguientes pasos

Toda la logica esta encapsulada en `src/` siguiendo SOLID; este notebook orquesta el pipeline.

## 1. Configuracion y carga de datos

In [ ]:
import sys
import json
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import Image, display

warnings.filterwarnings('ignore')
PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from src.config import Config
from src.data_loader import DataLoader
from src.preprocessing import ClientePreprocessor, TransaccionalPreprocessor, quality_report
from src.feature_engineering import TransactionalFeatureBuilder, FeatureAssembler
from src.eda import EDAReport
from src.model import ModelTrainer
from src.evaluation import ModelEvaluator, BusinessImpactAnalyzer
from src.strategy import StrategyBuilder
from src.visualization import Visualizer

cfg = Config()
viz = Visualizer(
    output_dir=cfg.graficos_dir,
    dpi=cfg.figure_dpi,
    figsize=cfg.figure_size,
    palette_primary=cfg.palette_primary,
    palette_secondary=cfg.palette_secondary,
    palette_accent=cfg.palette_accent,
)
print('Project root:', PROJECT_ROOT)
print('Graficos out:', cfg.graficos_dir)

In [ ]:
loader = DataLoader(cfg.data_clientes_path, cfg.data_transaccional_path)
raw = loader.load()
print('Clientes:', raw.clientes.shape)
print('Transacciones:', raw.transacciones.shape)
raw.clientes.head()

In [ ]:
raw.transacciones.head()

## 2. Calidad de datos y preprocesamiento

Validamos nulos, duplicados y tipos antes de avanzar.

In [ ]:
qr_cli = quality_report(raw.clientes)
qr_trans = quality_report(raw.transacciones)
print('Calidad clientes:', qr_cli)
print('Calidad transacciones:', qr_trans)

In [ ]:
cli_prep = ClientePreprocessor(age_strategy='median')
clientes_clean = cli_prep.fit_transform(raw.clientes)
trans_prep = TransaccionalPreprocessor(excel_date_origin=cfg.excel_date_origin)
trans_clean = trans_prep.transform(raw.transacciones)
print('Post limpieza clientes:', clientes_clean.shape)
print('Post limpieza transacciones:', trans_clean.shape)
print('Periodo:', trans_clean['date'].min(), 'a', trans_clean['date'].max())

## 3. Analisis exploratorio (EDA)

Distribucion del target, tasas por territorio y segmento, efecto de iniciativas comerciales y comportamiento transaccional.

In [ ]:
eda = EDAReport(target_col=cfg.target_col)
td = eda.target_distribution(clientes_clean)
print('Distribucion target:', td)
viz.target_balance(clientes_clean[cfg.target_col])
display(Image(str(cfg.graficos_dir / '01_target_balance.png')))

In [ ]:
print('Target por territorio:')
display(eda.target_rate_by_group(clientes_clean, 'territory_id'))
viz.target_rate_by_category(clientes_clean, 'territory_id')
display(Image(str(cfg.graficos_dir / '02_target_rate_by_territory_id.png')))

In [ ]:
print('Target por segmento:')
display(eda.target_rate_by_group(clientes_clean, 'segment'))
viz.target_rate_by_category(clientes_clean, 'segment')
display(Image(str(cfg.graficos_dir / '02_target_rate_by_segment.png')))

In [ ]:
viz.initiatives_vs_target(clientes_clean)
display(Image(str(cfg.graficos_dir / '03_initiatives_vs_target.png')))
viz.transactions_over_time(trans_clean)
display(Image(str(cfg.graficos_dir / '04_transactions_over_time.png')))
viz.category_amount(trans_clean)
display(Image(str(cfg.graficos_dir / '05_category_amount.png')))

## 4. Feature engineering transaccional

A partir de 392K transacciones construimos features RFM, monetarias, comportamentales, de diversidad (entropia / HHI), share por categoria y tendencia (primera vs segunda mitad del periodo).

In [ ]:
fb = TransactionalFeatureBuilder()
trans_features = fb.build(trans_clean)
print('Features transaccionales:', trans_features.shape)
master = FeatureAssembler().assemble(clientes_clean, trans_features)
print('Tabla maestra:', master.shape)
master.head()

In [ ]:
numeric_features = [c for c in master.columns if c not in (cfg.business_categorical + [cfg.id_col, cfg.target_col]) and master[c].dtype != object]
corr = eda.correlation_with_target(master, numeric_features)
display(corr.head(15))
viz.correlation_with_target(corr, top=15)
display(Image(str(cfg.graficos_dir / '06_corr_top_features.png')))

## 5. Modelado y validacion cruzada

Entrenamos 5 modelos (LR, RF, GB, XGB, LGBM) con `StratifiedKFold(5)` y ROC-AUC como metrica principal. Usamos `class_weight=balanced` o `scale_pos_weight` para manejar el desbalance (16.3% positivos).

In [ ]:
X = master[cfg.business_categorical + numeric_features]
y = master[cfg.target_col].astype(int)
trainer = ModelTrainer(
    categorical_features=cfg.business_categorical,
    numeric_features=numeric_features,
    random_state=cfg.random_state,
    test_size=cfg.test_size,
    cv_folds=5,
)
X_train, X_test, y_train, y_test = trainer.split(X, y)
cv_scores = trainer.cross_validate_models(X_train, y_train)
pd.DataFrame(cv_scores, index=['ROC-AUC mean', 'ROC-AUC std']).T

In [ ]:
viz.cv_scores(cv_scores)
display(Image(str(cfg.graficos_dir / '07_cv_scores.png')))
best_name = max(cv_scores, key=lambda k: cv_scores[k][0])
print('Mejor modelo:', best_name)
best_pipeline = trainer.fit_final(best_name, X_train, y_train)

## 6. Evaluacion del modelo final

ROC-AUC, PR-AUC, matriz de confusion, curvas ROC y PR, lift y curva de ganancias acumuladas.

In [ ]:
y_proba_test = best_pipeline.predict_proba(X_test)[:, 1]
evaluator = ModelEvaluator()
best_thr = evaluator.optimal_f1_threshold(y_test.values, y_proba_test)
m_default = evaluator.evaluate(y_test.values, y_proba_test, threshold=0.5)
m_optimal = evaluator.evaluate(y_test.values, y_proba_test, threshold=best_thr)
print('Umbral default (0.5):', {k: getattr(m_default, k) for k in ['roc_auc','pr_auc','accuracy','precision','recall','f1']})
print('Umbral optimo F1:', best_thr, {k: getattr(m_optimal, k) for k in ['accuracy','precision','recall','f1']})

In [ ]:
fpr, tpr = evaluator.roc_points(y_test.values, y_proba_test)
viz.roc_curve_plot(fpr, tpr, m_default.roc_auc)
display(Image(str(cfg.graficos_dir / '08_roc_curve.png')))
rec, prec = evaluator.pr_points(y_test.values, y_proba_test)
viz.pr_curve_plot(rec, prec, m_default.pr_auc)
display(Image(str(cfg.graficos_dir / '09_pr_curve.png')))
viz.confusion_matrix_plot(m_optimal.confusion)
display(Image(str(cfg.graficos_dir / '10_confusion_matrix.png')))

In [ ]:
deciles = evaluator.decile_analysis(y_test.values, y_proba_test)
display(deciles)
viz.lift_chart(deciles)
display(Image(str(cfg.graficos_dir / '11_lift_chart.png')))
viz.cumulative_gains(deciles)
display(Image(str(cfg.graficos_dir / '12_cumulative_gains.png')))

In [ ]:
try:
    cat_names = list(
        best_pipeline.named_steps['preprocessor']
        .named_transformers_['cat']
        .get_feature_names_out(cfg.business_categorical)
    )
    feature_names = numeric_features + cat_names
    step = best_pipeline.named_steps['model']
    if hasattr(step, 'feature_importances_'):
        importance = pd.Series(step.feature_importances_, index=feature_names)
    else:
        importance = pd.Series(np.abs(step.coef_[0]), index=feature_names)
    importance.sort_values(ascending=False).head(15).to_frame('importancia')
    viz.feature_importance(importance, top=15)
    display(Image(str(cfg.graficos_dir / '13_feature_importance.png')))
except Exception as e:
    print('No fue posible extraer importancia:', e)

## 7. Analisis de impacto economico

Valor esperado por cliente usando las tasas oficiales (10% potencial / 15% potencial + iniciativa / 0.5% sin potencial). Comparamos accionar vs no accionar por decil.

In [ ]:
y_proba_full = best_pipeline.predict_proba(X)[:, 1]
business = BusinessImpactAnalyzer(
    incremental_potencial=cfg.incremental_potencial,
    incremental_potencial_iniciativa=cfg.incremental_potencial_iniciativa,
    incremental_no_potencial=cfg.incremental_no_potencial,
)
business_df = business.deciles_business_value(master, y_proba_full, amount_col='total_amount')
display(business_df)
viz.business_impact(business_df)
display(Image(str(cfg.graficos_dir / '14_business_impact.png')))

## 8. Estrategia comercial por tier

Del score continuo a una segmentacion accionable en 4 tiers (A premium, B estandar, C nurturing, D mantenimiento).

In [ ]:
strategy = StrategyBuilder()
master_scored = master.copy()
master_scored['score'] = y_proba_full
master_scored['tier'] = strategy.assign_tier(y_proba_full)
tier_summary = strategy.tier_summary(master_scored, score_col='score', amount_col='total_amount')
display(tier_summary)
viz.tier_distribution(tier_summary)
display(Image(str(cfg.graficos_dir / '15_tier_distribution.png')))

In [ ]:
master_scored[[cfg.id_col, 'score', 'tier', 'target']].to_csv(cfg.outputs_dir / 'predicciones_clientes.csv', index=False)
print('Predicciones guardadas en', cfg.outputs_dir / 'predicciones_clientes.csv')

## 9. Conclusiones y siguientes pasos

**Conclusiones del modelo**
- Se construyo un pipeline modular (SOLID) que parte de datos crudos y entrega scores accionables por cliente.
- Se generaron 40+ features transaccionales (RFM, monetary, diversidad, share por categoria, tendencia).
- Se evaluaron 5 modelos via 5-fold stratified CV, eligiendo el de mejor ROC-AUC.
- El analisis por decil cuantifica el valor economico esperado bajo escenarios con/sin accion comercial.

**Estrategia recomendada**
- Concentrar inversion (Cliente Perfecto + Mercaderismo) en Tier A y B (top deciles), donde el uplift esperado es mas alto.
- Tier C: campanas de bajo costo (descuento focalizado, comunicacion).
- Tier D: operacion regular, sin inversion adicional.

**Siguientes pasos**
1. Test A/B controlado: asignar iniciativas al Tier A/B y medir uplift real vs el esperado.
2. Aumentar ventana temporal de transacciones (idealmente 12 meses) para capturar estacionalidad.
3. Incorporar features externos: ubicacion GIS, NSE de la zona, competencia local.
4. Migrar el pipeline a un orquestador (Airflow / Prefect) y servir scores en batch mensual.
5. Monitoreo: drift de features, recalibracion trimestral del modelo.
6. Explicabilidad: integrar SHAP a nivel cliente para que el comercial entienda por que un cliente es target.